# FLUKE Sentiment Analysis with OpenAI o3-2025-04-16 Reasoning Model

Consolidated notebook with unified utilities for consistency across all FLUKE o3 experiments.

In [ ]:
# Standard imports
from datasets import load_dataset
import dspy
import openai
import os
import pandas as pd
import json
import glob
import time
from dotenv import load_dotenv
from dspy.evaluate import Evaluate

# Import unified FLUKE utilities
from fluke_o3_utils import (
    REASONING_MODELS, REASONING_CONFIGS,
    remove_space, extract_classification_prediction,
    aggregate_results, highlight_drops_and_significance,
    compare_models
)

In [ ]:
# Load environment variables
load_dotenv()
openai.api_key = os.getenv('OPENAI_API_KEY')
openai.organization = os.getenv('OPENAI_ORGANIZATION')

## Configuration

In [ ]:
# Select configuration
CONFIG_NAME = 'standard'  # Options: 'standard', 'detailed', 'efficient'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy
lm = dspy.LM(MODEL_ID)
dspy.configure(lm=lm)

## Load Data

In [ ]:
# Load SST-2 dataset
ds = load_dataset('stanfordnlp/sst2')['validation']
print(f"Dataset size: {len(ds)}")

# Create examples
examples = [
    dspy.Example({
        "text": remove_space(r["sentence"]),
        "label": r["label"]
    }).with_inputs("text")
    for r in ds
]

# Test example
example = examples[835]
print(f"\nExample text: {example.text}")
print(f"Label: {example.label}")

## Define Task

In [ ]:
class O3Sentiment(dspy.Signature):
    """Classify sentiment of the given text. Think step by step and analyze the emotional tone, word choice, and overall sentiment. Answer with 1 for positive sentiment, 0 for negative sentiment."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class O3SentimentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(O3Sentiment)

    def forward(self, text):
        return self.prog(text=text)

# Initialize module
o3_sentiment = O3SentimentModule()

# Evaluation metric
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    parsed_answer = extract_classification_prediction(pred)
    return parsed_answer == str(true.label)

## Evaluate Original Dataset

In [ ]:
# Evaluate subset due to o3 costs
TEST_SIZE = 100  # Adjust based on budget
test_examples = examples[:TEST_SIZE]

print(f"Evaluating {len(test_examples)} examples...")

evaluate = Evaluate(
    devset=test_examples,
    metric=eval_metric,
    num_threads=1,  # Single thread for o3
    display_progress=True,
    display_table=10,
    return_outputs=True,
    return_all_scores=True
)

results = evaluate(o3_sentiment)

# Save results
items = []
for sample in results[1]:
    items.append({
        'text': sample[0]['text'],
        'label': sample[0]['label'],
        'pred': extract_classification_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']
    })

df_result = pd.DataFrame(items)
output_file = f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv'
df_result.to_csv(output_file, index=False)

print(f"\nAccuracy: {results[0]:.3f}")
print(f"Results saved to: {output_file}")

## Evaluate Modifications

In [ ]:
def evaluate_modified_set(data, program, max_samples=30):
    """Evaluate on modified dataset."""
    limited_data = data[:max_samples] if len(data) > max_samples else data
    
    mod_examples = [
        dspy.Example({
            "text": remove_space(r['modified_text']),
            "original_text": remove_space(r['original_text']),
            "label": int(r.get('modified_label', r['label'])),
            "original_label": int(r['label'])
        }).with_inputs("text")
        for r in limited_data
    ]
    
    evaluate = Evaluate(
        devset=mod_examples,
        metric=eval_metric,
        num_threads=1,
        display_progress=True,
        display_table=1,
        return_outputs=True,
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Load original predictions
original_pred_file = f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
    print(f"Loaded original predictions from {original_pred_file}")
else:
    print("Please run original evaluation first")
    original_pred_ds = None

# Test key modifications
test_modifications = ['typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json']
json_files = glob.glob('../data/modified_data/sa/*_100.json')
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"\nTesting modifications: {[f.split('/')[-1] for f in json_files]}")

for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    results_mod = evaluate_modified_set(data, o3_sentiment, max_samples=25)
    
    # Process results
    items = []
    for sample in results_mod[1]:
        item = {
            'text': sample[0]['text'],
            'original_text': sample[0]['original_text'],
            'modified_label': sample[0]['label'],
            'original_label': sample[0]['original_label'],
            'modified_pred': extract_classification_prediction(sample[1]['label']),
            'raw_output': sample[1]['label']
        }
        
        # Find original prediction
        if original_pred_ds is not None:
            matches = original_pred_ds[original_pred_ds['text'] == item['original_text']]
            item['original_pred'] = matches.iloc[0]['pred'] if not matches.empty else None
        else:
            item['original_pred'] = None
        
        items.append(item)
    
    df_mod = pd.DataFrame(items)
    mod_name = json_file.split('/')[-1].replace('.json', '')
    output_file = f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
    df_mod.to_csv(output_file, index=False)
    
    print(f"Accuracy: {results_mod[0]:.3f}")
    print(f"Saved to: {output_file}")
    
    time.sleep(5)  # Rate limiting

## Aggregate Results

In [ ]:
# Aggregate all modification results
result_files = glob.glob(f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')

if result_files:
    results_df = aggregate_results(
        result_files, 
        task_name='sentiment_analysis',
        model_name=f'{MODEL_NAME}-{CONFIG_NAME}'
    )
    
    if not results_df.empty:
        # Display summary
        print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
        print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])
        
        # Save aggregated results
        output_file = f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-DP.csv'
        results_df.to_csv(output_file, index=False)
        print(f"\nAggregated results saved to: {output_file}")
        
        # Display styled results
        styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
        display(styled_df)
else:
    print("No result files found to aggregate")

## Model Comparison

In [ ]:
# Compare with other models
comparison_files = {
    'GPT-4o': 'results/sa/gpt4o-0shot-sst2.csv',
    'Claude-3.5': 'results/sa/claude-3-5-sonnet-0shot-sst2.csv',
    'Mixtral-8x22B': 'results/sa/mixtral-8x22b-sst2.csv',
    f'{MODEL_NAME}-{CONFIG_NAME}': f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv'
}

comparison_df = compare_models(comparison_files, task_name='sentiment_analysis')

if not comparison_df.empty:
    print("\nModel Comparison:")
    print(comparison_df)
    
    # Highlight best performer
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]
    
    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No comparison data available")

## Summary

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Sentiment Analysis with {MODEL_NAME} Complete!")
print(f"{'='*60}")

if 'results' in locals():
    print(f"\nBase accuracy: {results[0]:.3f}")

if 'results_df' in locals() and not results_df.empty:
    avg_row = results_df[results_df['modification'] == 'average'].iloc[0]
    print(f"Average robustness drop: {avg_row['difference']:.3f}")
    print(f"Modifications tested: {len(results_df) - 1}")

print(f"\nConfiguration used: {config['description']}")
print(f"Files saved in: results/sa/")